# Import Library

In [ ]:
import pandas as pd
import requests
import time
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# Scraping the Data

## Film Link Grabbing
Pada tahap pertama, dilakukan proses scraping untuk memperoleh URL dari film-film tersebut. Karena informasi detail film berada pada halaman masing-masing film, proses diawali dengan mengumpulkan seluruh URL film yang tersedia.

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "en-US,en;q=0.9"
}

movies = []

for page in range(1, 100):
    url = f"https://www.themoviedb.org/movie/top-rated?page={page}"
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    links = soup.select('a[data-media-type="movie"][href^="/movie/"]')
    for link in links:
        if link.find("h2"):
            full = urljoin("https://www.themoviedb.org", link["href"])
            movies.append(full)
    print(f"Page {page} completed")
    time.sleep(1)

movie_links = list(dict.fromkeys(movies))
df = pd.DataFrame({"movie_url": movie_links})
df.to_csv("tmdb_movie_links.csv", index=False)
print(f"\nCollected {len(movie_links)} unique movie links")

Page 1 completed
Page 2 completed
Page 3 completed
Page 4 completed
Page 5 completed
Page 6 completed
Page 7 completed
Page 8 completed
Page 9 completed
Page 10 completed
Page 11 completed
Page 12 completed
Page 13 completed
Page 14 completed
Page 15 completed
Page 16 completed
Page 17 completed
Page 18 completed
Page 19 completed
Page 20 completed
Page 21 completed
Page 22 completed
Page 23 completed
Page 24 completed
Page 25 completed
Page 26 completed
Page 27 completed
Page 28 completed
Page 29 completed
Page 30 completed
Page 31 completed
Page 32 completed
Page 33 completed
Page 34 completed
Page 35 completed
Page 36 completed
Page 37 completed
Page 38 completed
Page 39 completed
Page 40 completed
Page 41 completed
Page 42 completed
Page 43 completed
Page 44 completed
Page 45 completed
Page 46 completed
Page 47 completed
Page 48 completed
Page 49 completed
Page 50 completed
Page 51 completed
Page 52 completed
Page 53 completed
Page 54 completed
Page 55 completed
Page 56 completed
P

## Grabbing Detailed Data
Pada tahap kedua, CSV yang dibuat pada step 1 dibaca kembali untuk mendapatkan seluruh URL film yang telah dikumpulkan pada tahap sebelumnya. Setiap URL kemudian diakses satu per satu untuk mengambil informasi detail yang tersedia pada halaman film.

In [ ]:
links = pd.read_csv("tmdb_movie_links.csv")

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html, */*; q=0.01",
    "X-Requested-With": "XMLHttpRequest"
}

data = []

for link in links["movie_url"]:
    response = requests.get(link, headers=headers, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    # Judul film
    judul = None
    title = soup.select_one("div.title h2 a")

    if title:
        judul = title.get_text(strip=True)

    print(f"Scraping {judul}")

    # Tahun rilis
    tahun = None
    year = soup.select_one("div.title span.release_date")

    if year:
        tahun = year.get_text(strip=True).strip("()")

    # Panjang film
    waktu = None
    runtime = soup.select_one("div.facts span.runtime")

    if runtime:
        text = runtime.get_text(" ", strip=True)
        hours = 0
        minutes = 0
        # Konversi ke angka beneran biar tidak ribet
        if "h" in text:
            parts = text.split("h")
            hours = int(parts[0].strip())
        if "m" in text:
            parts = text.split("h")
            minutes = int(parts[-1].replace("m", "").strip())
        waktu = (hours*60) + minutes

    # Genre film
    genre1 = None
    genre2 = None
    genre = soup.select("div.facts span.genres a")

    if len(genre) >= 1:
        genre1 = genre[0].get_text(strip=True)
    if len(genre) >= 2:
        genre2 = genre[1].get_text(strip=True) 

    # Director film
    director = None
    profiles = soup.select("ol.people li.profile")

    for profile in profiles:
        roles = profile.select_one("p.character")
        if roles:
            role = roles.get_text(" ",strip=True)
            if "Director" in role:
                name = profile.select_one("p a")
                if name:
                    director = name.get_text(strip=True)
                break


    # Skor dan jumlah user yang rating
    skor = None
    rate = None

    # btw, ini dilakukan karena informasi rating pada website tidak langsung tersedia pada halaman utama film (wajib melakukan interaksi pada bagian score)
    rating_link = (link.rstrip("/") + "/remote/rating/details?translate=false")
    rating_response = requests.get(rating_link, headers, timeout=30)
    rating_response.raise_for_status()
    rating_soup = BeautifulSoup(rating_response.text, "html.parser")
    score = rating_soup.select_one("div.user_score_chart[data-percent]")
    ratings = rating_soup.select_one("h3")

    if score:
        skor = int(score["data-percent"])

    if ratings:
        rate = int(ratings.get_text(strip=True).split()[0].replace(",", ""))

    # Budget dan Revenue
    budget = None
    revenue = None

    facts = soup.select("section.facts.left_column p")

    for fact in facts:
        factos = fact.get_text(" ",strip=True)
        if factos.startswith("Budget"):
            budget = factos.replace("Budget","",1).strip()
        if factos.startswith("Revenue"):
            revenue = factos.replace("Revenue","",1).strip()

    data.append({
        "title": judul,
        "year": tahun,
        "runtime": waktu,
        "genre_1": genre1,
        "genre_2": genre2,
        "director": director,
        "score": skor,
        "user_ratings": rate,
        "budget": budget,
        "revenue": revenue,
    })

    time.sleep(1)

movies_df = pd.DataFrame(data)
movies_df["year"] = pd.to_numeric(movies_df["year"])
movies_df["runtime"] = pd.to_numeric(movies_df["runtime"])
movies_df["score"] = pd.to_numeric(movies_df["score"])
movies_df["user_ratings"] = pd.to_numeric(movies_df["user_ratings"])
movies_df.to_csv("tmdb_movie_database.csv", index=False)

print("\nScraping completed!")
print(f"Movies scraped: {len(movies_df)}")

Scraping Avatar Aang: The Last Airbender
Scraping Swapped
Scraping Accidental Partners
Scraping Facing El Chapo
Scraping Demon Slayer: Kimetsu no Yaiba Infinity Castle
Scraping The Shawshank Redemption
Scraping The Godfather
Scraping Michael
Scraping Project Hail Mary
Scraping The Godfather Part II
Scraping 12 Angry Men
Scraping Schindler's List
Scraping Young Hearts
Scraping Chainsaw Man - The Movie: Reze Arc
Scraping The Dark Knight
Scraping Spirited Away
Scraping The Green Mile
Scraping The Lord of the Rings: The Return of the King
Scraping Dilwale Dulhania Le Jayenge
Scraping Parasite
Scraping Interstellar
Scraping Your Name.
Scraping Pulp Fiction
Scraping The Good, the Bad and the Ugly
Scraping Forrest Gump
Scraping Harakiri
Scraping Selena Gomez: My Mind & Me
Scraping GoodFellas
Scraping Seven Samurai
Scraping Grave of the Fireflies
Scraping ¿Quieres ser mi hijo?
Scraping The Lord of the Rings: The Fellowship of the Ring
Scraping Life Is Beautiful
Scraping Fight Club
Scraping Hum